In [ ]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

# EDA

In [ ]:
from seqeval.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    AutoTokenizer
)
from datasets import Dataset
import json
import numpy as np

In [ ]:
with open('../../data/synthetic/train.json', 'r') as f:
    data = json.load(f)
with open('../../data/synthetic/val.json', 'r') as f:
    eval_data = json.load(f)

In [ ]:
print(type(data))
print(data.keys())

In [ ]:
data['label_order']

In [ ]:
print(type(data['examples']))
print(f'data length: {len(data['examples'])}')

In [ ]:
print(f'{type(data['examples'][0])}')
print(f'length of 1 data: {len(data['examples'][0])}')
print(f'keys of 1 data: {data['examples'][0].keys()}')

In [ ]:
print(data['examples'][0]['tokens'])
print()
print(data['examples'][0]['labels'])

# Preprocessing

In [ ]:
examples = data['examples']
eval_examples = eval_data['examples']

In [ ]:
label2id = {
    label : i for i,label in enumerate(data['label_order'])
}
id2label = {
    i : label for i,label in enumerate(data['label_order'])
}
label2id

## tokenizing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

In [ ]:
tokens = examples[0]['tokens']

encoding = tokenizer(tokens, is_split_into_words = True)
print(encoding.tokens())

In [ ]:
# ngetes pake kalimat for better understanding

# tokens = 'Jl. Betonmas Selatan no. 185'

# encoding = tokenizer(tokens, is_split_into_words = False)
# print(encoding.tokens())

In [ ]:
print(encoding.word_ids())

In [ ]:
# create dataset object

dataset = Dataset.from_list(examples)

eval_dataset = Dataset.from_list(eval_examples)

In [ ]:
def tokenize_align_labels(example):
    tokens = example["tokens"]
    labels = example["labels"]
    
    encoding = tokenizer(
        tokens,
        is_split_into_words = True,
        truncation=True,
        max_length=512
    )

    word_ids = encoding.word_ids()

    aligned_labels = []

    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
           aligned_labels.append(-100)

        elif word_id != previous_word_id:
            aligned_labels.append(
                label2id[labels[word_id]]
            )
        else:
            aligned_labels.append(-100)


        previous_word_id = word_id

    encoding['labels'] = aligned_labels

    return encoding

In [ ]:
# apply the whole preprocessing pipeline
tokenized_dataset = dataset.map(tokenize_align_labels)
eval_tokenized_dataset = eval_dataset.map(tokenize_align_labels)

In [ ]:
tokenized_dataset

# Compute Metrics

In [ ]:
def compute_metrics(eval_preds):

    predictions, labels = eval_preds

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred_labels = []
        true_label = []

        for pred, lab in zip(prediction, label):

            # Ignore special tokens and padding
            if lab == -100:
                continue

            pred_labels.append(id2label[pred])
            true_label.append(id2label[lab])

        true_predictions.append(pred_labels)
        true_labels.append(true_label)

    return {
        "accuracy": accuracy_score(
            true_labels,
            true_predictions
        ),
        "precision": precision_score(
            true_labels,
            true_predictions
        ),
        "recall": recall_score(
            true_labels,
            true_predictions
        ),
        "f1": f1_score(
            true_labels,
            true_predictions
        )
    }

# training

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=len(data['label_order']),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    logging_steps=50,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    compute_metrics = compute_metrics,
    eval_dataset = eval_tokenized_dataset
)

trainer.train()

# test

In [ ]:
with open('../../data/synthetic/test.json', 'r') as f:
    test_data = json.load(f)

test_examples = test_data['examples']
test_dataset = Dataset.from_list(test_examples)
test_tokenized_dataset = test_dataset.map(tokenize_align_labels)

In [ ]:
trainer.evaluate(test_tokenized_dataset)

# save model

In [ ]:
# model.save_pretrained("/kaggle/working/bert_alamatin", from_pt=True) 